# UpliftBench: end-to-end on Kaggle (Criteo Uplift v2)

Self-contained notebook that trains all five estimators on a **2M-row stratified subsample**
(to stay under the 9-hour Kaggle CPU wall) and renders the Qini comparison.

Repo: https://github.com/TirtheshJani/UpliftBench

## Install

In [ ]:
!pip install -q lightgbm causalml econml dowhy pyarrow 'networkx<3.3' typer joblib

## Download Criteo Uplift v2

In [ ]:
from pathlib import Path

import requests

DATA = Path("/kaggle/working/criteo.csv.gz")
if not DATA.exists():
    URL = "https://criteo-uplift.s3.amazonaws.com/criteo-uplift-v2.1.csv.gz"
    with requests.get(URL, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(DATA, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
print(DATA.stat().st_size / 1e9, "GB")

## Subsample to 2M rows (stratified by treatment)

In [ ]:
import numpy as np
import pandas as pd

FEATURES = [f"f{i}" for i in range(12)]
DTYPES = {
    **{f: "float32" for f in FEATURES},
    "treatment": "uint8",
    "exposure": "uint8",
    "visit": "uint8",
    "conversion": "uint8",
}
df = pd.read_csv(DATA, dtype=DTYPES)
rng = np.random.default_rng(42)
n_target = 2_000_000
p_t = df["treatment"].mean()
n_t = int(round(n_target * p_t))
n_c = n_target - n_t
idx = np.concatenate(
    [
        rng.choice(df.index[df["treatment"] == 1], size=n_t, replace=False),
        rng.choice(df.index[df["treatment"] == 0], size=n_c, replace=False),
    ]
)
df = df.loc[np.sort(idx)].reset_index(drop=True)
print(len(df), df["treatment"].mean(), df["visit"].mean())

## Train/test split (random, NOT stratified by T)

In [ ]:
perm = rng.permutation(len(df))
n_test = int(round(len(df) * 0.2))
test_idx = np.sort(perm[:n_test])
train_idx = np.sort(perm[n_test:])
X_train = df.loc[train_idx, FEATURES].to_numpy(np.float32)
X_test = df.loc[test_idx, FEATURES].to_numpy(np.float32)
T_train = df.loc[train_idx, "treatment"].to_numpy(np.uint8)
T_test = df.loc[test_idx, "treatment"].to_numpy(np.uint8)
Y_train = df.loc[train_idx, "visit"].to_numpy(np.uint8)
Y_test = df.loc[test_idx, "visit"].to_numpy(np.uint8)

## Helpers: Qini and AUUC

In [ ]:
def qini_curve(t, y, cate):
    order = np.argsort(-cate, kind="mergesort")
    ts = t[order].astype(float)
    ys = y[order].astype(float)
    cum_t = ts.cumsum()
    cum_c = (1 - ts).cumsum()
    cum_yt = (ys * ts).cumsum()
    cum_yc = (ys * (1 - ts)).cumsum()
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.where(cum_c > 0, cum_t / cum_c, 0.0)
    lift = cum_yt - cum_yc * ratio
    n = len(t)
    return np.concatenate(([0.0], np.arange(1, n + 1) / n)), np.concatenate(([0.0], lift))


def qini_coef(t, y, cate):
    xs, ys = qini_curve(t, y, cate)
    q = ys[-1]
    if abs(q) < 1e-12:
        return 0.0
    return float(2 * (np.trapezoid(ys, xs) - q / 2) / abs(q))

## Train five estimators

In [ ]:
import lightgbm as lgb

LGB = dict(
    n_estimators=200,
    num_leaves=63,
    max_bin=63,
    learning_rate=0.05,
    min_data_in_leaf=200,
    verbose=-1,
)


def fit_classifier(X, Y):
    m = lgb.LGBMClassifier(**LGB)
    m.fit(X, Y)
    return m


def fit_regressor(X, Y):
    m = lgb.LGBMRegressor(**LGB)
    m.fit(X, Y)
    return m

In [ ]:
# S-learner
Xs = np.column_stack([X_train, T_train.astype(np.float32)])
s = lgb.LGBMClassifier(**LGB)
s.fit(Xs, Y_train)


def cate_s(X):
    return (
        s.predict_proba(np.column_stack([X, np.ones(len(X), np.float32)]))[:, 1]
        - s.predict_proba(np.column_stack([X, np.zeros(len(X), np.float32)]))[:, 1]
    )


cate_s_test = cate_s(X_test)
qs = qini_coef(T_test, Y_test, cate_s_test)
print("S-learner Qini", qs)

In [ ]:
# T-learner
m1 = fit_classifier(X_train[T_train == 1], Y_train[T_train == 1])
m0 = fit_classifier(X_train[T_train == 0], Y_train[T_train == 0])
cate_t_test = m1.predict_proba(X_test)[:, 1] - m0.predict_proba(X_test)[:, 1]
qt = qini_coef(T_test, Y_test, cate_t_test)
print("T-learner Qini", qt)

In [ ]:
# X-learner (LightGBM from scratch)
mu0 = fit_classifier(X_train[T_train == 0], Y_train[T_train == 0])
mu1 = fit_classifier(X_train[T_train == 1], Y_train[T_train == 1])
d1 = Y_train[T_train == 1] - mu0.predict_proba(X_train[T_train == 1])[:, 1]
d0 = mu1.predict_proba(X_train[T_train == 0])[:, 1] - Y_train[T_train == 0]
tau1 = fit_regressor(X_train[T_train == 1], d1)
tau0 = fit_regressor(X_train[T_train == 0], d0)
g = T_train.mean()
cate_x_test = g * tau0.predict(X_test) + (1 - g) * tau1.predict(X_test)
qx = qini_coef(T_test, Y_test, cate_x_test)
print("X-learner Qini", qx)

In [ ]:
# DR-learner
from econml.dr import DRLearner

dr = DRLearner(
    model_propensity=lgb.LGBMClassifier(**LGB),
    model_regression=lgb.LGBMRegressor(**LGB),
    model_final=lgb.LGBMRegressor(**LGB),
    cv=3,
)
dr.fit(Y=Y_train.astype(float), T=T_train.astype(int), X=X_train)
cate_dr_test = dr.effect(X_test).ravel()
qdr = qini_coef(T_test, Y_test, cate_dr_test)
print("DR-learner Qini", qdr)

In [ ]:
# DML
from econml.dml import LinearDML

dml = LinearDML(
    model_y=lgb.LGBMRegressor(**LGB),
    model_t=lgb.LGBMClassifier(**LGB),
    discrete_treatment=True,
    cv=3,
)
dml.fit(Y=Y_train.astype(float), T=T_train.astype(int), X=X_train)
cate_dml_test = dml.effect(X_test).ravel()
qdml = qini_coef(T_test, Y_test, cate_dml_test)
print("DML Qini", qdml)

## Comparison

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5))
for name, cate in [
    ("s-learner", cate_s_test),
    ("t-learner", cate_t_test),
    ("x-learner", cate_x_test),
    ("dr-learner", cate_dr_test),
    ("dml", cate_dml_test),
]:
    xs, ys = qini_curve(T_test, Y_test, cate)
    ax.plot(xs, ys, label=f"{name} ({qini_coef(T_test, Y_test, cate):+.3f})")
ax.set_xlabel("Population fraction")
ax.set_ylabel("Cumulative uplift")
ax.legend()
ax.set_title("Qini curves on 2M sample (Criteo Uplift v2)")
fig